 # Whisper Accent Robustness — Model Performance Evaluation



 Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.



 - **Scripted**     → 6 held-out test speakers (never seen during training)

 - **Spontaneous**  → suitcase corpus (OOD, all speakers)



 Metrics:

 - **WER**  — word error rate (primary ASR metric)

 - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"

# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline":      "Zero-shot",
    "no_aux_heldout_chinese": "Naive LoRA FT (h/o Chinese)",
    "no_aux":        "Naive LoRA FT",
    # "no_aux_heldout_chinese":        "Naive LoRA FT (heldout Chinese)",
    # "ctc_aux_l3":    "CTC Aux",
    # "feat_aux":      "Feat Aux",
    # "feat_aux_heldout_chinese":      "Feat Aux (heldout Chinese)",
    # "feat_aux0p3":      "Feat Aux (0.3)",
    # "both_aux":      "CTC + Feat",

    "whisfusion": "WhisFusion (Zero-shot)",
    "whisfusion_ft": "WhisFusion (LoRA FT)",
    "whisfusion_ft_hoc": "WhisFusion (LoRA FT, heldout Chinese)",
}
SPLITS = ["scripted", "spontaneous"]


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from jiwer import wer as jiwer_wer

pd.set_option("display.max_colwidth", 80)


In [3]:
def load_results(model_key: str, split: str) -> pd.DataFrame | None:
    p = Path(RESULTS_DIR) / f"{model_key}_{split}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)

# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, dict[str, pd.DataFrame | None]] = {}
for key in MODEL_KEYS:
    results[key] = {}
    for split in SPLITS:
        df = load_results(key, split)
        results[key][split] = df
        if df is not None:
            print(f"  loaded  {key}/{split}: {len(df):,} rows")


  loaded  baseline/scripted: 7,638 rows
  loaded  baseline/spontaneous: 527 rows
  loaded  no_aux_heldout_chinese/scripted: 7,638 rows
  loaded  no_aux_heldout_chinese/spontaneous: 527 rows
  loaded  no_aux/scripted: 7,638 rows
  loaded  no_aux/spontaneous: 527 rows
  loaded  whisfusion/scripted: 7,638 rows
  loaded  whisfusion/spontaneous: 527 rows
  loaded  whisfusion_ft/scripted: 7,638 rows
  loaded  whisfusion_ft/spontaneous: 527 rows
  loaded  whisfusion_ft_hoc/scripted: 7,638 rows
  [missing] results/model_perf_comparison/whisfusion_ft_hoc_spontaneous_predictions.csv — run eval_model_perf.py first


In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def available(key: str, split: str) -> bool:
    return results.get(key, {}).get(split) is not None


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer_wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))


def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived), precomputed by eval_model_perf.py."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer_wer(
                            sub["reference_norm"].fillna("").tolist(),
                            sub["prediction_norm"].fillna("").tolist(),
                        )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")


Helpers loaded.


 ---

 # Part 1 — Overall WER & PER (G2P)

In [5]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if not available(key, split):
            continue
        df  = results[key][split]
        wer = corpus_wer(df)
        per = corpus_per(df)
        rows.append({"Model": label, "Split": split, "WER": wer, "PER (G2P)": per})

overall_df = pd.DataFrame(rows)

for metric in ["WER", "PER (G2P)"]:
    pivot = overall_df.pivot(index="Model", columns="Split", values=metric)
    display(
        pivot.style
             .format("{:.4f}")
             .background_gradient(cmap="RdYlGn_r", axis=0)
             .set_caption(f"{metric} by Model × Split (lower is better)")
    )


Split,scripted,spontaneous
Model,,
Naive LoRA FT,0.0731,0.2683
Naive LoRA FT (h/o Chinese),0.0782,0.2638
WhisFusion (LoRA FT),0.0832,0.8300
"WhisFusion (LoRA FT, heldout Chinese)",0.1264,nan
WhisFusion (Zero-shot),0.2688,0.4137
Zero-shot,0.1625,0.2063


Split,scripted,spontaneous
Model,,
Naive LoRA FT,0.0452,0.2442
Naive LoRA FT (h/o Chinese),0.0484,0.2347
WhisFusion (LoRA FT),0.0658,0.6999
"WhisFusion (LoRA FT, heldout Chinese)",0.0993,nan
WhisFusion (Zero-shot),0.1536,0.3086
Zero-shot,0.0871,0.1779


In [6]:
from plotly.subplots import make_subplots

# ── Separate WER/PER into side-by-side subplots, one figure per split ─────────

for split in SPLITS:
    sub = overall_df[overall_df["Split"] == split].copy()
    if sub.empty:
        continue

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("WER", "PER (G2P)"),
        shared_yaxes=True,
    )

    # Left: WER
    wer_vals = sub["WER"]
    fig.add_trace(
        go.Bar(
            name="WER",
            x=sub["Model"].tolist(),
            y=wer_vals.tolist(),
            text=[f"{v:.1%}" if pd.notna(v) else "" for v in wer_vals],
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=1
    )

    # Right: PER
    per_vals = sub["PER (G2P)"]
    if per_vals.notna().any():
        fig.add_trace(
            go.Bar(
                name="PER (G2P)",
                x=sub["Model"].tolist(),
                y=per_vals.tolist(),
                text=[f"{v:.1%}" if pd.notna(v) else "" for v in per_vals],
                textposition="outside",
                showlegend=False,
            ),
            row=1, col=2
        )

    fig.update_yaxes(title_text="Error Rate", tickformat=".0%", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", row=1, col=2)
    fig.update_layout(
        title=f"WER vs PER by Model — {split.capitalize()}",
        margin=dict(t=90),
    )
    fig.show()


 ---

 # Part 2 — Per-L1 WER

In [7]:
for split in SPLITS:
    l1_rows = []
    for key, label in MODEL_KEYS.items():
        if not available(key, split):
            continue
        grp = grouped_wer(results[key][split], "l1")
        grp["Model"] = label
        l1_rows.append(grp)
    if not l1_rows:
        continue

    l1_df = pd.concat(l1_rows, ignore_index=True)

    # Delta vs zero-shot baseline (negative = improvement)
    base_label = MODEL_KEYS["no_aux"]
    base_wer   = (
        l1_df[l1_df["Model"] == base_label][["l1", "wer"]]
        .rename(columns={"wer": "wer_base"})
    )
    l1_df = l1_df.merge(base_wer, on="l1", how="left")
    l1_df["wer_delta_pct"] = (
        (l1_df["wer"] - l1_df["wer_base"]) / l1_df["wer_base"] * 100
    )

    l1s = sorted(l1_df["l1"].unique())

    # Explicit model order (left -> right inside each L1 group)
    model_order = [
        "baseline",
        "no_aux",
        "no_aux_heldout_chinese",
        "whisfusion",
        "whisfusion_ft",
        "whisfusion_ft_hoc",
    ]

    # Keep only models present in MODEL_KEYS/results for this split
    ordered_items = [
        (k, MODEL_KEYS[k])
        for k in model_order
        if k in MODEL_KEYS and available(k, split)
    ]

    fig = go.Figure()
    for key, label in ordered_items:
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name=label,
            x=l1s,
            y=[sub.loc[l, "wer"] if l in sub.index else None for l in l1s],
            text=[f"{sub.loc[l, 'wer']:.1%}" if l in sub.index else "" for l in l1s],
            textposition="outside",
        ))

    fig.update_layout(
        title=f"WER by L1 — {split.capitalize()}",
        barmode="group",
        yaxis=dict(title="WER", tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin=dict(t=80),
    )
    fig.show()

    out = Path(RESULTS_DIR) / f"comparison_{split}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")
    display(
        l1_df.sort_values(["l1", "Model"])
             .style.format({
                 "wer":           "{:.4f}",
                 "wer_base":      "{:.4f}",
                 "wer_delta_pct": "{:+.1f}%",
                 "per":           lambda v: f"{v:.4f}" if pd.notna(v) else "—",
             })
             .set_caption(f"{split.capitalize()} — Per-L1 WER")
    )


Saved results/model_perf_comparison/comparison_scripted_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
14,Arabic,974,0.1084,0.0718,Naive LoRA FT,0.1084,+0.0%
7,Arabic,974,0.1154,0.0750,Naive LoRA FT (h/o Chinese),0.1084,+6.5%
28,Arabic,974,0.1356,0.1149,WhisFusion (LoRA FT),0.1084,+25.2%
35,Arabic,974,0.1907,0.1556,"WhisFusion (LoRA FT, heldout Chinese)",0.1084,+76.0%
21,Arabic,974,0.3747,0.2245,WhisFusion (Zero-shot),0.1084,+245.7%
0,Arabic,974,0.2388,0.1375,Zero-shot,0.1084,+120.4%
15,Chinese,1130,0.0910,0.0543,Naive LoRA FT,0.0910,+0.0%
8,Chinese,1130,0.1107,0.0687,Naive LoRA FT (h/o Chinese),0.0910,+21.6%
29,Chinese,1130,0.0838,0.0636,WhisFusion (LoRA FT),0.0910,-8.0%
36,Chinese,1130,0.1398,0.1089,"WhisFusion (LoRA FT, heldout Chinese)",0.0910,+53.6%


Saved results/model_perf_comparison/comparison_spontaneous_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
16,Arabic,11,0.1735,0.0942,Naive LoRA FT,0.1735,+0.0%
8,Arabic,11,0.1588,0.0816,Naive LoRA FT (h/o Chinese),0.1735,-8.5%
32,Arabic,11,0.8294,0.7600,WhisFusion (LoRA FT),0.1735,+378.0%
24,Arabic,11,0.2559,0.1431,WhisFusion (Zero-shot),0.1735,+47.5%
0,Arabic,11,0.2118,0.1627,Zero-shot,0.1735,+22.0%
17,Chinese,19,0.2396,0.1311,Naive LoRA FT,0.2396,+0.0%
9,Chinese,19,0.2733,0.1365,Naive LoRA FT (h/o Chinese),0.2396,+14.0%
33,Chinese,19,0.8119,0.7195,WhisFusion (LoRA FT),0.2396,+238.8%
25,Chinese,19,0.3050,0.1640,WhisFusion (Zero-shot),0.2396,+27.3%
1,Chinese,19,0.2238,0.1308,Zero-shot,0.2396,-6.6%


 ---

 # Part 3 — Scripted vs Spontaneous Gap (zero-shot)



 Different corpora → compared at L1 level, not speaker level.

In [8]:
if available("baseline", "scripted") and available("baseline", "spontaneous"):
    s_g  = grouped_wer(results["baseline"]["scripted"],    "l1").rename(columns={"wer": "WER_scripted"})
    sp_g = grouped_wer(results["baseline"]["spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    gap  = s_g[["l1", "WER_scripted"]].merge(
               sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
    gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]

    l1s = gap["l1"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Scripted",    x=l1s, y=gap["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
    fig.update_layout(
        title   = "Zero-shot WER — Scripted vs Spontaneous by L1",
        barmode = "group",
        yaxis   = dict(title="WER", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()
    display(
        gap.style
           .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
           .set_caption("Scripted vs Spontaneous WER gap (zero-shot)")
    )


,l1,WER_scripted,WER_spontaneous,gap
0,Arabic,0.2388,0.2118,-0.0270
1,Chinese,0.1916,0.2238,0.0322
2,English,0.0382,0.1688,0.1307
3,Hindi,0.0653,0.0924,0.0271
4,Korean,0.0805,0.1844,0.1039
5,Spanish,0.2284,0.1343,-0.0941
6,Vietnamese,0.3117,0.2484,-0.0633


 ---

 # Part 4 — Utterance-level Analysis

In [9]:
# ── UTT WER distributions ─────────────────────────────────────────────────────
fig = go.Figure()
key = "whisfusion_ft"
label = MODEL_KEYS[key]
split = "scripted"
fig.add_trace(go.Histogram(
    x       = results[key][split]["utt_wer"],
    name    = f"{label} / {split}",
    opacity = 0.5,
    nbinsx  = 40,
))
fig.update_layout(
    title    = "Utterance WER Distribution — All Conditions",
    barmode  = "overlay",
    xaxis    = dict(title="Utterance WER"),
    yaxis    = dict(title="Count"),
    legend   = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
)
fig.show()


In [10]:
# ── Worst utterances per model — scripted, cross-model comparison ─────────────
split    = "spontaneous"
N_WORST  = 30
models = [
    # "baseline",
    "no_aux",
    # "feat_aux0p3",
    "whisfusion_ft",
]
base_df = results["no_aux"][split]

for anchor_key in models:
    if not available(anchor_key, split):
        continue

    anchor_df = results[anchor_key][split]
    idx       = anchor_df.nlargest(N_WORST, "utt_wer").index

    worst = base_df.loc[idx, ["utterance_id", "speaker", "l1", "reference_norm"]].copy()

    # Add each model's prediction + WER + PER for these utterances
    for key in models:
        if not available(key, split):
            continue
        other            = results[key][split]
        col              = key
        worst[f"pred_{col}"] = other.loc[idx, "prediction_norm"].values
        worst[f"wer_{col}"]  = other.loc[idx, "utt_wer"].values
        if "utt_per" in other.columns:
            worst[f"per_{col}"] = other.loc[idx, "utt_per"].values

    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(("wer_", "per_"))}
    display(
        worst.style
                .format(fmt_cols)
                .set_caption(f"Top-{N_WORST} Worst Utterances for {MODEL_KEYS[anchor_key]} — {split.capitalize()}")
    )

,utterance_id,speaker,l1,reference_norm,pred_no_aux,wer_no_aux,per_no_aux,pred_whisfusion_ft,wer_whisfusion_ft,per_whisfusion_ft
180,EDACC-C49-000000031,EDACC-C49-A,Jamaican,nah its not cold one thing though that i miss about being home like i cant wait to go back home in september is like the,and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened and then all that stuff happened,17.038,20.000,nan,1.000,1.000
364,EDACC-C03-000000061,EDACC-C03-B,English,rotating circular thing icon,we were taking exactly the open icon,1.500,0.857,we wereoting even she in anon,1.500,0.905
288,EDACC-C49-000000203,EDACC-C49-A,Jamaican,you gotta be tripping,youve got to be sure phil,1.250,0.571,youllt be your,0.750,0.643
229,EDACC-C49-000000110,EDACC-C49-B,Jamaican,its like july the eighteenth so,id buy the right for a fee so,1.167,0.737,by by the right as being,1.000,0.842
119,EDACC-C48-000000022,EDACC-C48-A,Jamaican,yeah i think,yea i think i think,1.000,0.857,ye i i i,1.000,0.714
155,EDACC-C48-000000089,EDACC-C48-A,Jamaican,yeah they decided they wanted to stop it,yea that is an irate subject,1.000,0.741,yea he i iron,1.000,0.852
190,EDACC-C49-000000054,EDACC-C49-B,Jamaican,its really fun though,the fairly summed down,1.000,0.917,the very so so,1.000,0.833
242,EDACC-C49-000000131,EDACC-C49-A,Jamaican,youd like to go where,you dancing over it,1.000,0.846,you yout go,0.800,0.692
285,EDACC-C49-000000199,EDACC-C49-A,Jamaican,is it going to,its a coin,1.000,0.800,it,0.750,0.800
294,EDACC-C49-000000210,EDACC-C49-B,Jamaican,three hundred dollars thats a,hes on his eyes,1.000,0.850,i you eyes thatge,1.000,0.800


,utterance_id,speaker,l1,reference_norm,pred_no_aux,wer_no_aux,per_no_aux,pred_whisfusion_ft,wer_whisfusion_ft,per_whisfusion_ft
364,EDACC-C03-000000061,EDACC-C03-B,English,rotating circular thing icon,we were taking exactly the open icon,1.500,0.857,we wereoting even she in anon,1.500,0.905
449,EDACC-C06-000000160,EDACC-C06-A,English,fever dreamy details,beavadreamy details,0.667,0.400,at dream me dis,1.333,0.600
197,EDACC-C49-000000062,EDACC-C49-B,Jamaican,thats like a zombie,thats like a song,0.250,0.385,that i i look me,1.250,0.615
309,EDACC-C49-000000229,EDACC-C49-A,Jamaican,cause its just grease,its just green,0.500,0.286,it it is just just greed,1.250,0.643
478,EDACC-C43_P1-000000083,EDACC-C43-B,English,um whatevers going if theres leftovers or a sandwich,um whatevers going if theres leftovers or sandwich,0.111,0.027,and what was was gone in is sth is wch her,1.222,0.838
27,hkk_chunk2,HKK,Korean,not see each other they just bumped into each other and um and especially for the man he dropped his glasses um after bumping into each other um,we did not see each other they just bumped into each other and especially for the men he dropped his classes after bumping into each other,0.286,0.174,nan,1.000,1.000
91,ydck_chunk9,YDCK,Korean,uh fallen on the floor but maybe after a few seconds uh a few minutes they uh they pull they are pulling themselves uh themselves together meaning the man,fallen on the floor maybe after a few seconds a few minutes they they pull theyre pulling themselves themselves together meaning the men,0.276,0.113,ol upon upon two two,1.000,0.918
110,EDACC-C48-000000005,EDACC-C48-B,Jamaican,um i i cannot remember playing any games at high school any games that we wouldve played might have been on our phones like um i dont whats that one where you jump over the trains,i cannot remember playing any games at high school any games i would have played might have been on our phones like whats that one where you jump over the trains,0.222,0.150,nan,1.000,1.000
119,EDACC-C48-000000022,EDACC-C48-A,Jamaican,yeah i think,yea i think i think,1.000,0.857,ye i i i,1.000,0.714
122,EDACC-C48-000000026,EDACC-C48-A,Jamaican,yeah i dont i dont remember why we stopped either because i remember that were doing it from that time and then we just stop at one point so i know that when we before i know that we stopped before im not sure we stopped when we got to grade nine but im sure that we stopped before we got into grade ten before where were in skirt and blouse i know we stopped before,yea i dont i dont remember why we stopped at it because i remember that were doing it from that time and then we just stop at one point so i know that when we before i know that we stop before im not sure who stopped when weve got to agree them but im sure that we stop before we go into grade 10 before where were in skirt and nose and always stop before,0.211,0.123,nan,1.000,1.000


In [11]:
# ── Utterance-level head-to-head win counts ───────────────────────────────────
split = "scripted"   # change to "spontaneous" if needed

available_keys = [k for k in MODEL_KEYS if available(k, split)]

for anchor_key in models:
    anchor_label = MODEL_KEYS[anchor_key]
    anchor_wer   = results[anchor_key][split]["utt_wer"].fillna(1.0)
    n_utts       = len(anchor_wer)

    print(f"\n{anchor_label} vs others ({split}, n={n_utts} utterances):")
    for other_key in models:
        if other_key == anchor_key:
            continue
        other_wer  = results[other_key][split]["utt_wer"].fillna(1.0)
        anchor_label_short = MODEL_KEYS[other_key]

        wins  = (anchor_wer < other_wer).sum()
        ties  = (anchor_wer == other_wer).sum()
        loses = (anchor_wer > other_wer).sum()

        print(f"  vs {anchor_label_short:<20} | "
              f"wins={wins:>4} ({wins/n_utts:.1%})  "
              f"ties={ties:>4} ({ties/n_utts:.1%})  "
              f"loses={loses:>4} ({loses/n_utts:.1%})")


Naive LoRA FT vs others (scripted, n=7638 utterances):
  vs WhisFusion (LoRA FT) | wins=1787 (23.4%)  ties=4598 (60.2%)  loses=1253 (16.4%)

WhisFusion (LoRA FT) vs others (scripted, n=7638 utterances):
  vs Naive LoRA FT        | wins=1253 (16.4%)  ties=4598 (60.2%)  loses=1787 (23.4%)


# Held-out L1 Performance - Accent-Robustness

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path("output")
outdir.mkdir(exist_ok=True)

held_out_l1 = "English"
split = "spontaneous"
model_keys = [
    "baseline",
    "no_aux",
    "no_aux_heldout_chinese",
    # "feat_aux",
    "whisfusion",
    "whisfusion_ft",
    "whisfusion_ft_hoc",
]

rows = []
for key in model_keys:
    if key not in results or split not in results[key]:
        continue

    df = results[key][split].copy()
    if "l1" not in df.columns:
        continue

    df = df[df["l1"] == held_out_l1]
    if len(df) == 0:
        continue

    row = {
        "model_key": key,
        "model": MODEL_KEYS.get(key, key),
        "n_utts": len(df),
        "n_speakers": df["speaker"].nunique() if "speaker" in df.columns else None,
        "wer_mean": corpus_wer(df) * 100,
    }

    if "utt_per" in df.columns:
        row["per_mean"] = df["utt_per"].mean()
        row["per_std"] = df["utt_per"].std()

    rows.append(row)

heldout_cmp = pd.DataFrame(rows).reset_index(drop=True)
heldout_cmp.to_csv(outdir / "heldout_l1_comparison.csv", index=False)

plot_df = heldout_cmp.copy()
plot_df["label"] = plot_df["model"] if "model" in plot_df.columns else plot_df["model_key"]

metrics = [("wer_mean", "WER (%)", "#01696f")]
# if "per_mean" in plot_df.columns and plot_df["per_mean"].notna().any():
#     metrics.append(("per_mean", "per_std", "PER", "#0b5177"))

fig, axes = plt.subplots(len(metrics), 1, figsize=(12, 8 * len(metrics)), sharex=True)
if len(metrics) == 1:
    axes = [axes]

x = range(len(plot_df))
for ax, (mean_col, title, color) in zip(axes, metrics):
    means = plot_df[mean_col].to_numpy()
    ax.bar(x, means, color=color, alpha=0.7, capsize=5)
    ax.set_ylabel(title)
    ax.set_title(f"Held-out L1 comparison — {held_out_l1} / {split} — {title} mean ± std")
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

axes[-1].set_xticks(list(x))
axes[-1].set_xticklabels(plot_df["label"], rotation=25, ha="right")
axes[-1].set_xlabel("Model")

fig.tight_layout()
fig.savefig(outdir / "heldout_l1_points_whiskers.png", dpi=200, bbox_inches="tight")
fig.show()
print(outdir / "heldout_l1_comparison.csv")
print(outdir / "heldout_l1_points_whiskers.png")

fmt = {
    c: "{:.3f}"
    for c in heldout_cmp.columns
    if c.startswith(("wer_", "per_"))
}


display(
    heldout_cmp.style
        .format(fmt)
        .set_caption(f"Held-out L1 comparison — {held_out_l1} / {split}")
)

AttributeError: 'NoneType' object has no attribute 'copy'

In [ ]:
import numpy as np
import pandas as pd

def align_for_sigtest(
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    group_value: str | None = None,
    group_col: str = "l1",
    metric_col: str = "utt_wer",
) -> pd.DataFrame:
    a = df_a.copy()
    b = df_b.copy()

    if group_value is not None:
        a = a[a[group_col] == group_value].copy()
        b = b[b[group_col] == group_value].copy()

    cols_a = ["utterance_id", group_col, metric_col]
    cols_b = ["utterance_id", group_col, metric_col]

    merged = (
        a[cols_a]
        .rename(columns={metric_col: "score_a", group_col: f"{group_col}_a"})
        .merge(
            b[cols_b].rename(columns={metric_col: "score_b", group_col: f"{group_col}_b"}),
            on="utterance_id",
            how="inner",
        )
    )

    if group_value is None and f"{group_col}_a" in merged.columns:
        merged[group_col] = merged[f"{group_col}_a"]

    merged = merged.dropna(subset=["score_a", "score_b"]).copy()
    merged["diff"] = merged["score_a"] - merged["score_b"]   # positive => model B better
    return merged